# JAXA extraction bounds

Divides each JAXA Vietnam 2020 v23.09 tile extent into four bounding boxes.
Use [`replication/prepare_inputs.py`](replication/prepare_inputs.py) for the maintained bounds workflow and [`replication/prepare_crop_mask.py`](replication/prepare_crop_mask.py) for mosaicking.

## Setup and output

Start Jupyter from the repository root with GDAL Python bindings.
Set `input_dir` to `datasets/Vietnam_v23.09_10m/` or the source tile directory.
The formula assumes north-up, square-cell WGS84 GeoTIFFs without rotation and uses x resolution for both axes.
The maintained helper validates these assumptions.

`JAXA_subtiles_full.json` maps `NxxExxx_i_j` to `[west, south, east, north]` in degrees.
`i` increases eastward and `j` southward.
Each box spans half the tile in both directions, without requiring integer-pixel partitions for odd dimensions.
This notebook neither splits pixels nor filters rice; later extraction selects class 3 from `b1`.

A prior run with all 60 source tiles and an explicit input path produced 240 boxes in a temporary directory, checked against source-raster bounds.
Old outputs and execution counts were cleared.
See [`NOTEBOOKS.md`](NOTEBOOKS.md) for workflow and verification commands.


## 1. Set the input directory

`pathlib` handles Windows and Unix filename separators.


In [1]:
from pathlib import Path
from osgeo import gdal

input_dir = Path('datasets/Vietnam_v23.09_10m')

## 2. Construct bounding boxes

Reads metadata in sorted tile order without loading pixels.
Preserves the geographic arithmetic and west/east then north/south loop order.


In [2]:
# Getting all available tile paths
raster_list = [str(path) for path in input_dir.glob('*.tif')]
raster_list.sort()

tiles_dict = dict()

# For each tile
for raster_path in raster_list:

    raster = gdal.Open(raster_path)
    gt = raster.GetGeoTransform()

    raster_name = Path(raster_path).name[:7]


    # Getting raster dimensions
    xmin = gt[0]
    ymax = gt[3]
    res = gt[1]
    xlen = res * raster.RasterXSize
    ylen = res * raster.RasterYSize

    # Split each tile into four equal sized subtiles
    div = 2
    xsize = xlen / div
    ysize = ylen / div

    xstep =[xmin + xsize * i for i in range(div+1)]
    ystep =[ymax - ysize * i for i in range(div+1)]

    # Redefining each subtile's bounds
    for i in range(div):
        for j in range(div):
            xmin = xstep[i]
            xmax = xstep[i+1]
            ymax = ystep[j]
            ymin = ystep[j+1]

            # appending new subtile and its bounds to the dict
            tile_name = raster_name+'_'+str(i)+'_'+str(j)
            tiles_dict[tile_name] = [xmin, ymin, xmax, ymax]

## 3. Preview bounds

Shows four boxes and the total count.


In [3]:
print(f'{len(tiles_dict)} subtiles')
for key, bounds in list(tiles_dict.items())[:4]:
    print(key, bounds)

240 subtiles
N08E104_0_0 [104.0, 8.5, 104.5, 9.0]
N08E104_0_1 [104.0, 8.0, 104.5, 8.5]
N08E104_1_0 [104.5, 8.5, 105.0, 9.0]
N08E104_1_1 [104.5, 8.0, 105.0, 8.5]


## 4. Save bounds

Overwrites `JAXA_subtiles_full.json` with extraction extents, not administrative or surveyed boundaries.


In [4]:
# Export to JSON file
import json
with open('JAXA_subtiles_full.json', 'w') as fp:
    json.dump(tiles_dict, fp)